# RAG 실습 — 빈 템플릿

**학생 실습용** — 청킹부터 하이브리드 검색까지 직접 구현하세요.

## 목표
1. 텍스트 청킹 함수 작성
2. VoyageAI 임베딩 생성
3. ChromaDB 벡터 검색 구현
4. BM25 검색 구현
5. 하이브리드 검색 (RRF) 통합
6. Claude 연동 RAG 질의응답

In [ ]:
# ── Setup ──────────────────────────────────────────────
import math
import re
from collections import Counter

import anthropic
import chromadb
import numpy as np
import voyageai
from dotenv import load_dotenv

load_dotenv()

claude_client = anthropic.Anthropic()
voyage_client = voyageai.Client()
MODEL = "claude-haiku-4-5"

## 샘플 문서

아래 문서를 RAG 파이프라인으로 검색 가능하게 만드세요.

In [ ]:
DOCUMENT = """
콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 고강도 콘크리트의 경우 fck 40 MPa 이상을 적용할 수 있다. 철근의 항복강도 fy는 400 MPa 또는 500 MPa를 표준으로 한다.

RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다. 최대 철근비는 균형 철근비의 0.75배를 초과할 수 없다. 전단보강은 스터럽 간격이 d/2 이하가 되도록 배치해야 한다.

기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 4개이며, 최소 철근비는 0.01 이상이어야 한다.

고정하중은 구조물 자체의 무게와 영구적으로 부착된 부분의 무게를 포함한다. 콘크리트의 단위중량은 24 kN/m3, 철근콘크리트는 25 kN/m3을 표준값으로 한다.

적재하중은 건축물의 용도에 따라 다르게 적용한다. 주거용 건물의 바닥 적재하중은 2.0 kN/m2, 사무실은 2.5 kN/m2, 상점은 4.0 kN/m2를 적용한다.
""".strip()

## Task 1: 텍스트 청킹 함수 작성

크기 기반 청킹 함수를 완성하세요.

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """TODO: 크기 기반 청킹 함수를 구현하세요.
    
    힌트:
    - start 위치에서 chunk_size만큼 잘라냅니다
    - start를 (chunk_size - overlap)만큼 이동합니다
    - text 끝까지 반복합니다
    """
    chunks = []
    # ── 여기에 코드 작성 ──
    
    # ── 여기까지 ──
    return chunks


# 테스트
chunks = chunk_text(DOCUMENT)
print(f"생성된 청크 수: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  청크 {i}: {len(c)}자 — {c[:50]}...")

## Task 2: VectorIndex 클래스 구현

`add_documents`와 `search` 메서드를 완성하세요.

In [ ]:
class VectorIndex:
    def __init__(self, collection_name="practice"):
        self.voyage_client = voyageai.Client()
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.create_collection(
            name=collection_name, metadata={"hnsw:space": "cosine"}
        )

    def add_documents(self, documents, ids=None):
        """TODO: 문서를 임베딩하고 ChromaDB에 저장하세요.
        
        힌트:
        - voyage_client.embed()로 임베딩 생성 (input_type="document")
        - collection.add()로 ChromaDB에 저장
        """
        if ids is None:
            ids = [f"doc_{i}" for i in range(len(documents))]
        # ── 여기에 코드 작성 ──
        
        # ── 여기까지 ──

    def search(self, query, top_k=3):
        """TODO: 쿼리를 임베딩하고 유사 문서를 검색하세요.
        
        힌트:
        - voyage_client.embed()로 쿼리 임베딩 (input_type="query")
        - collection.query()로 검색
        - [{"text": ..., "score": ..., "id": ...}] 형태로 반환
        """
        # ── 여기에 코드 작성 ──
        
        # ── 여기까지 ──
        return []

## Task 3: BM25Index 클래스 구현

`search` 메서드를 완성하세요.

In [ ]:
class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.documents = []
        self.doc_lengths = []
        self.avg_doc_length = 0
        self.doc_freqs = Counter()
        self.doc_term_freqs = []

    def add_documents(self, documents):
        self.documents = documents
        for doc in documents:
            tokens = doc.lower().split()
            self.doc_lengths.append(len(tokens))
            tf = Counter(tokens)
            self.doc_term_freqs.append(tf)
            for t in set(tokens):
                self.doc_freqs[t] += 1
        self.avg_doc_length = sum(self.doc_lengths) / len(self.doc_lengths)

    def search(self, query, top_k=3):
        """TODO: BM25 점수를 계산하고 상위 K개를 반환하세요.
        
        힌트:
        - IDF = log((N - df + 0.5) / (df + 0.5) + 1)
        - BM25 = IDF * tf * (k1+1) / (tf + k1*(1-b+b*dl/avgdl))
        """
        # ── 여기에 코드 작성 ──
        
        # ── 여기까지 ──
        return []

## Task 4: 하이브리드 검색 (Retriever + RRF)

두 검색 결과를 RRF로 통합하는 Retriever를 구현하세요.

In [ ]:
class Retriever:
    def __init__(self, vector_index, bm25_index):
        self.vector_index = vector_index
        self.bm25_index = bm25_index

    def search(self, query, top_k=3):
        """TODO: 하이브리드 검색을 구현하세요.
        
        힌트:
        1. vector_index.search()와 bm25_index.search() 각각 실행
        2. RRF 점수 계산: score += 1/(60 + rank)
        3. 통합 결과를 점수 기준 정렬
        """
        # ── 여기에 코드 작성 ──
        
        # ── 여기까지 ──
        return []

## Task 5: Claude 연동 RAG 질의응답

검색된 컨텍스트로 Claude에게 질문하세요.

In [ ]:
def rag_query(question, retriever, top_k=3):
    """TODO: 하이브리드 RAG 질의응답을 구현하세요.
    
    힌트:
    1. retriever.search()로 관련 청크 검색
    2. 검색된 텍스트를 system 프롬프트에 포함
    3. claude_client.messages.create()로 답변 생성
    """
    # ── 여기에 코드 작성 ──
    
    # ── 여기까지 ──
    return ""


# 테스트 (Task 1~4를 모두 완성한 후 실행)
# answer = rag_query("RC 보의 최소 철근비는?", retriever)
# print(answer)

## 도전 과제

1. 구조 기반 청킹도 추가하여 비교해보세요
2. `top_k`를 1, 3, 5로 바꾸며 답변 품질을 비교해보세요
3. 한국어 토크나이저를 개선해보세요 (konlpy 등 활용)